In [1]:
import numpy as np
import pandas as pd
from paraphrase_detection import data_shuffle_split, SBERTPairClassifier, Train, TextPairDataset, log_metrics_and_model
import torch
from sentence_transformers import SentenceTransformer
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
from loguru import logger
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

/Users/oli.dmrs/Documents/Personal Projects/paraphrase-detection-ml/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_parquet("hf://datasets/sentence-transformers/quora-duplicates/pair-class/train-00000-of-00001.parquet")
data = df.to_numpy()
data = data[:5000]

X = data[:, :2]
y = data[:, 2:3]

X_train, X_val, X_test, y_train, y_val, y_test = data_shuffle_split(X, y, 0.15, 0.12, 42)

In [3]:
seed = 42
model = SentenceTransformer("all-MiniLM-L6-v2")
fc_layer_sizes = [model.get_sentence_embedding_dimension()]
dropout_p = 0.1
lr = 1e-3
epochs = 10
batch_size = 32
steps = epochs * X_train.shape[0] // batch_size 
n_warmup_steps = steps * 0.1
n_freeze = 3
weight_decay = 1e-2

np.random.seed(seed)
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
model = SBERTPairClassifier(model, fc_layer_sizes, device, dropout_p)
optimizer = torch.optim.AdamW(model.parameters(), lr, weight_decay = weight_decay)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps = n_warmup_steps, num_training_steps = steps)
criterion = torch.nn.CrossEntropyLoss()

In [5]:
train_loader = DataLoader(dataset = TextPairDataset(X_train, y_train), batch_size = batch_size, shuffle = True, num_workers = 4)
validation_loader = DataLoader(dataset = TextPairDataset(X_val, y_val), batch_size = batch_size, shuffle = True, num_workers = 4)
test_loader = DataLoader(dataset = TextPairDataset(X_test, y_test), batch_size = batch_size, shuffle = True, num_workers = 4)

In [5]:
train = Train(model = model,
              optimizer = optimizer,
              scheduler = scheduler,
              criterion = criterion, 
              device = device,
              n_freeze = n_freeze,
              epochs = epochs,
              train_dataloader = train_loader,
              val_dataloader = validation_loader
              )

best_params, avg_batch_train_loss, epoch_train_acc, avg_batch_val_loss, epoch_val_acc, epoch_val_f1 = train.run_training_loop()

EPOCH: 0


2025-11-15 18:19:25.606 | INFO     | paraphrase_detection.train:run_training_loop:105 - Epoch 0: train loss = 0.7055100003878275
2025-11-15 18:19:25.608 | INFO     | paraphrase_detection.train:run_training_loop:106 - Epoch 0: validation loss = 0.677947461605072
2025-11-15 18:19:25.608 | INFO     | paraphrase_detection.train:run_training_loop:107 - Epoch 0: train acc = 0.32432432432432434
2025-11-15 18:19:25.609 | INFO     | paraphrase_detection.train:run_training_loop:108 - Epoch 0: validation acc = 1.0
2025-11-15 18:19:25.609 | INFO     | paraphrase_detection.train:run_training_loop:109 - Epoch 0: validation f1 = 1.0


EPOCH: 1


2025-11-15 18:19:51.088 | INFO     | paraphrase_detection.train:run_training_loop:105 - Epoch 1: train loss = 0.6643145084381104
2025-11-15 18:19:51.089 | INFO     | paraphrase_detection.train:run_training_loop:106 - Epoch 1: validation loss = 0.6082544922828674
2025-11-15 18:19:51.090 | INFO     | paraphrase_detection.train:run_training_loop:107 - Epoch 1: train acc = 0.7837837837837838
2025-11-15 18:19:51.090 | INFO     | paraphrase_detection.train:run_training_loop:108 - Epoch 1: validation acc = 0.7272727272727273
2025-11-15 18:19:51.091 | INFO     | paraphrase_detection.train:run_training_loop:109 - Epoch 1: validation f1 = 0.42105263157894735


EPOCH: 2


...

2025-11-15 18:22:24.393 | INFO     | paraphrase_detection.train:run_training_loop:105 - Epoch 7: train loss = 0.44610293706258136
2025-11-15 18:22:24.394 | INFO     | paraphrase_detection.train:run_training_loop:106 - Epoch 7: validation loss = 0.4654337167739868
2025-11-15 18:22:24.395 | INFO     | paraphrase_detection.train:run_training_loop:107 - Epoch 7: train acc = 0.7432432432432432
2025-11-15 18:22:24.395 | INFO     | paraphrase_detection.train:run_training_loop:108 - Epoch 7: validation acc = 0.7272727272727273
2025-11-15 18:22:24.396 | INFO     | paraphrase_detection.train:run_training_loop:109 - Epoch 7: validation f1 = 0.42105263157894735


EPOCH: 8


2025-11-15 18:22:49.902 | INFO     | paraphrase_detection.train:run_training_loop:105 - Epoch 8: train loss = 0.48721136649449664
2025-11-15 18:22:49.903 | INFO     | paraphrase_detection.train:run_training_loop:106 - Epoch 8: validation loss = 0.46390092372894287
2025-11-15 18:22:49.904 | INFO     | paraphrase_detection.train:run_training_loop:107 - Epoch 8: train acc = 0.7432432432432432
2025-11-15 18:22:49.904 | INFO     | paraphrase_detection.train:run_training_loop:108 - Epoch 8: validation acc = 0.7272727272727273
2025-11-15 18:22:49.905 | INFO     | paraphrase_detection.train:run_training_loop:109 - Epoch 8: validation f1 = 0.42105263157894735


EPOCH: 9


2025-11-15 18:23:15.444 | INFO     | paraphrase_detection.train:run_training_loop:105 - Epoch 9: train loss = 0.437816321849823
2025-11-15 18:23:15.445 | INFO     | paraphrase_detection.train:run_training_loop:106 - Epoch 9: validation loss = 0.46198713779449463
2025-11-15 18:23:15.445 | INFO     | paraphrase_detection.train:run_training_loop:107 - Epoch 9: train acc = 0.7432432432432432
2025-11-15 18:23:15.446 | INFO     | paraphrase_detection.train:run_training_loop:108 - Epoch 9: validation acc = 0.7272727272727273
2025-11-15 18:23:15.446 | INFO     | paraphrase_detection.train:run_training_loop:109 - Epoch 9: validation f1 = 0.42105263157894735


In [6]:
log_metrics_and_model('SBERTv1', avg_batch_train_loss, epoch_train_acc, avg_batch_val_loss, \
                      epoch_val_acc, epoch_val_f1, best_params)

In [6]:
freezes = [1, 3, 4, 5, 6]
for freeze in freezes:
    logger.info(f'STARTING training on freeze = {freeze}')
    train = Train(model = model,
                  optimizer = optimizer,
                  scheduler = scheduler,
                  criterion = criterion, 
                  device = device,
                  n_freeze = freeze,
                  epochs = epochs,
                  train_dataloader = train_loader,
                  val_dataloader = validation_loader
                  )

    best_params, avg_batch_train_loss, epoch_train_acc, avg_batch_val_loss, epoch_val_acc, epoch_val_f1 = train.run_training_loop() 
    log_metrics_and_model(f'SBERT_{freeze}freeze', avg_batch_train_loss, epoch_train_acc, avg_batch_val_loss, \
                          epoch_val_acc, epoch_val_f1)

2025-11-15 18:30:30.662 | INFO     | __main__:<module>:3 - STARTING training on freeze = 1


EPOCH: 0


2025-11-15 18:31:30.237 | INFO     | paraphrase_detection.train:run_training_loop:105 - Epoch 0: train loss = 0.554068723295489
2025-11-15 18:31:30.238 | INFO     | paraphrase_detection.train:run_training_loop:106 - Epoch 0: validation loss = 0.4173083882778883
2025-11-15 18:31:30.238 | INFO     | paraphrase_detection.train:run_training_loop:107 - Epoch 0: train acc = 0.6745989304812834
2025-11-15 18:31:30.238 | INFO     | paraphrase_detection.train:run_training_loop:108 - Epoch 0: validation acc = 0.7803921568627451
2025-11-15 18:31:30.238 | INFO     | paraphrase_detection.train:run_training_loop:109 - Epoch 0: validation f1 = 0.7635800731775964


EPOCH: 1


2025-11-15 18:32:27.999 | INFO     | paraphrase_detection.train:run_training_loop:105 - Epoch 1: train loss = 0.39267059243642366
2025-11-15 18:32:28.000 | INFO     | paraphrase_detection.train:run_training_loop:106 - Epoch 1: validation loss = 0.3908347934484482
2025-11-15 18:32:28.000 | INFO     | paraphrase_detection.train:run_training_loop:107 - Epoch 1: train acc = 0.8144385026737968
2025-11-15 18:32:28.001 | INFO     | paraphrase_detection.train:run_training_loop:108 - Epoch 1: validation acc = 0.8156862745098039
2025-11-15 18:32:28.001 | INFO     | paraphrase_detection.train:run_training_loop:109 - Epoch 1: validation f1 = 0.8028782894736842


...

2025-11-15 19:16:42.102 | INFO     | paraphrase_detection.train:run_training_loop:105 - Epoch 7: train loss = 0.15437666931722918
2025-11-15 19:16:42.103 | INFO     | paraphrase_detection.train:run_training_loop:106 - Epoch 7: validation loss = 0.4160646656528115
2025-11-15 19:16:42.103 | INFO     | paraphrase_detection.train:run_training_loop:107 - Epoch 7: train acc = 0.9526737967914438
2025-11-15 19:16:42.104 | INFO     | paraphrase_detection.train:run_training_loop:108 - Epoch 7: validation acc = 0.8058823529411765
2025-11-15 19:16:42.104 | INFO     | paraphrase_detection.train:run_training_loop:109 - Epoch 7: validation f1 = 0.7982812556182806


EPOCH: 8


2025-11-15 19:17:39.735 | INFO     | paraphrase_detection.train:run_training_loop:105 - Epoch 8: train loss = 0.1546898777795653
2025-11-15 19:17:39.736 | INFO     | paraphrase_detection.train:run_training_loop:106 - Epoch 8: validation loss = 0.42079807072877884
2025-11-15 19:17:39.737 | INFO     | paraphrase_detection.train:run_training_loop:107 - Epoch 8: train acc = 0.9508021390374332
2025-11-15 19:17:39.737 | INFO     | paraphrase_detection.train:run_training_loop:108 - Epoch 8: validation acc = 0.8176470588235294
2025-11-15 19:17:39.738 | INFO     | paraphrase_detection.train:run_training_loop:109 - Epoch 8: validation f1 = 0.8108029406719826


EPOCH: 9


2025-11-15 19:18:37.199 | INFO     | paraphrase_detection.train:run_training_loop:105 - Epoch 9: train loss = 0.15400854345315543
2025-11-15 19:18:37.200 | INFO     | paraphrase_detection.train:run_training_loop:106 - Epoch 9: validation loss = 0.4209194229915738
2025-11-15 19:18:37.200 | INFO     | paraphrase_detection.train:run_training_loop:107 - Epoch 9: train acc = 0.9510695187165775
2025-11-15 19:18:37.201 | INFO     | paraphrase_detection.train:run_training_loop:108 - Epoch 9: validation acc = 0.803921568627451
2025-11-15 19:18:37.201 | INFO     | paraphrase_detection.train:run_training_loop:109 - Epoch 9: validation f1 = 0.7960815673730508
